# Night Flash — Kaggle

Seven cells. Run them top to bottom; only cell 3 needs editing.

| cell | what it does |
|---|---|
| 1 | install and import |
| 2 | locate the attached Dataset, print image counts |
| 3 | **configuration — one `Session` object** |
| 4 | `auto_resume()` then `train` — read the briefing it prints |
| 5 | inference and previews |
| 6 | final snapshot |

The TUI does not run on Kaggle. `nightflash-tui` is for your Mac.

Full instructions, both-GPU setup, the command cookbook and the reasoning
behind each setting: **`docs/QuickStartKaggle.md`**.

### Before you start

1. Upload your photo folders as a Kaggle Dataset, keeping the two domains in
   separate subfolders: `normal_photos` (source) and `disposable_photos`
   (target). Add a `validation_photos` folder of held-out night-flash photos.
2. Attach it in the notebook's right-hand **Add Data** panel.
3. Accelerator: **GPU T4 x2**. A ~4M model will not saturate one T4, so the
   second GPU is best spent on a second experiment in a copy of this notebook.
   Do **not** use a TPU.

Check the target domain before you spend GPU hours on it — see
`docs/QuickStartKaggle.md` § *Before you start*. If `disposable_photos` holds
daylight photos, the model will learn the opposite of the intended look.

### Resuming across sessions

`/kaggle/working` is wiped when a session ends, so cell 3 sets
`snapshot_every=1`: a Dataset version is published after every epoch. Losing
the session then costs at most one epoch.

On the next session, attach **both** the data Dataset and the snapshot Dataset,
then run cells 1-4 again. `s.auto_resume()` finds the snapshot by name and
restores it. It prints `starting fresh` and does nothing if there is no
snapshot, which is exactly right for a first run.

Snapshots need Kaggle API credentials in `/root/.kaggle/kaggle.json`. Without
them you get a warning each epoch and training continues unaffected.

### Inference notes

Whole-frame is the default and preserves the exact original size. For
native-resolution detail use `s.infer(photo, tile=256)` — and keep `tile` equal
to `image_size`, because a mismatch is a distribution shift. Add
`max_edge=1024` to downscale the long edge while keeping the aspect ratio.

In [ ]:
# ── 1. install ──────────────────────────────────────────────────────
import subprocess, sys, os
from pathlib import Path

PROJECT = Path("/kaggle/working/nightflash-edge")
INPUT = Path("/kaggle/input")

# Find the code: a git URL, an unzipped copy, or your own dataset of the repo.
REPO_URL = os.environ.get("NIGHTFLASH_REPO_URL", "")
if REPO_URL and not PROJECT.exists():
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(PROJECT)], check=True)
if not (PROJECT / "pyproject.toml").exists():
    raise RuntimeError(
        "Put the repo at /kaggle/working/nightflash-edge. Either clone it in a "
        "cell above, or upload the repo as a Dataset and unzip it here."
    )
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", str(PROJECT)], check=True)
sys.path.insert(0, str(PROJECT / "src"))

import torch
from nightflash import Session
from nightflash.briefing import brief
from nightflash.notebook import show_previews, show_image
from nightflash.kaggle import environment_report, recommended_device

for line in ["kaggle:", *[f"  {k}: {v}" for k, v in environment_report().items()]]:
    print(line)

In [ ]:
# ── 2. data — attached Kaggle Datasets live under /kaggle/input ─────
def find_dataset_dir(must_contain: str) -> Path:
    """Locate the attached dataset that holds the requested subfolder."""
    for candidate in sorted(INPUT.iterdir()):
        if (candidate / must_contain).is_dir():
            return candidate
    raise FileNotFoundError(
        f"No attached dataset contains {must_contain!r}. "
        f"Attach it on the right-hand panel. Saw: {[p.name for p in INPUT.iterdir()]}"
    )

DATASET = find_dataset_dir("normal_photos")
RUNS = Path("/kaggle/working/runs")

SOURCE_DIR = DATASET / "normal_photos"
TARGET_DIR = DATASET / "disposable_photos"
VAL_DIR = DATASET / "validation_photos"
MASK_DIR = None  # set to DATASET/"source_masks" when you have Vision masks

for label, path in [("source", SOURCE_DIR), ("target", TARGET_DIR)]:
    if not path.is_dir():
        raise FileNotFoundError(f"{label} not found: {path}")
    n = sum(1 for p in path.rglob("*") if p.suffix.lower() in {".jpg", ".jpeg", ".png", ".webp"})
    print(f"{label:7s} {n:6d} images")
print("runs root (wiped each session):", RUNS)

In [ ]:
# ── 3. configuration — the only thing you normally edit ─────────────
s = Session(
    arch="edgehat",          # edgehat | swinir | restormer | hybrid
    head="freqsplit",        # tanh | perpixel | freqsplit | lut3d
                             #   | lut3d+freqsplit | lut3d+perpixel | tanh+freqsplit
    name="night",            # run folder suffix: edgehat_freqsplit_<name>
    source_dir=SOURCE_DIR,
    target_dir=TARGET_DIR,
    val_dir=VAL_DIR,
    mask_dir=MASK_DIR,
    runs_root=RUNS,
    image_size=256,          # keep --tile equal to this for tiled inference
    batch_size=8,
    epochs=50,               # ABSOLUTE target epoch, not a count of extra epochs
    device="auto",           # "auto" resolves to cuda on Kaggle/Colab, mps on Mac
    use_amp=True,
    num_workers=2,
    preview_every=250,
    log_every=25,
    halation=True,
    subject_gate=True,
    region_exposure=True,
    augment_preset="default",   # default | conservative | strong | none
    snapshot_every=1,        # publish a Dataset version every epoch
    # snapshot_dataset="my-slug",   # optional; defaults to nightflash-<run>
)
print(s)

print(brief(s))  # full pre-flight plan

In [ ]:
# ── 4. resume, then train ─────────────────────────────────────────
# On Kaggle, /kaggle/working is wiped between sessions, so the run is
# snapshotted to a Dataset version. Attach that Dataset in the Add Data panel
# of the new session before running this cell.
# Pull in the last snapshot automatically, or start fresh.
# Safe to run every session, and a no-op outside Kaggle.
s.auto_resume()

# Read this before training. It prints the plan, the flags, the paths, the
# flow, the current state, and a numbered decision menu.
print(brief(s, mode="train"))

# Pick ONE. Each line is idempotent and safe to re-run.
s.train(extra_epochs=20)        # relative: 20 more from wherever the run is
# s.train()                    # absolute: go to Session.epochs (cell 3)

In [ ]:
# ── 5. look at the output ─────────────────────────────────────────
# Pick ONE. Each line is idempotent and safe to re-run.

s.infer(f"{SOURCE_DIR}", seeds=2, save_comparison=True)   # whole frame, 1:1
# s.infer(photo, tile=256)                  # native detail; tile == image_size
# s.infer(photo, tile=256, tile_overlap=64) # faster, slightly more seams
# s.infer(photo, max_edge=1024)             # downscale long edge, keep ratio
# s.infer(photo, max_edge=1024, tile=256)  # downscale AND tile
# s.infer(photo, seeds=3)                   # 3 stochastic variants

show_previews(s.run_dir, n=4)
print("\nRun directory:", s.run_dir)
print("State:", s.state().status)
print("Inference output:", s.inference_dir)

In [ ]:
# ── 6. keep the run ──────────────────────────────────────────────
# Snapshot so the next session can resume. With snapshot_every=1 this already
# happened after every epoch; this is just a final labelled copy.
s.snapshot(note="final")

print("Run directory:", s.run_dir)
print("State:", s.state().status)
print("Checkpoint:", s.latest if s.latest.exists() else "(none yet)")

# Core ML export needs coremltools, which is easiest on macOS. Download the
# checkpoint and run:
#
#   python -m nightflash.export_coreml --checkpoint <here> --output model.mlpackage
#   python -m nightflash.validate_coreml --checkpoint <here> \
#       --coreml model.mlpackage --input photo.jpg
#
# Save Version with output enabled keeps /kaggle/working as a fallback.